In [1]:
# Data Cleaning - NOMIS Vacancy Data
# cleaning the raw nomis file ready for analysis

import pandas as pd

# loading the raw file
df = pd.read_excel('NOMIS_Vacancies_13Jun2026.xlsx', skiprows=6)

# fixing headers
df.columns = df.iloc[0]
df = df.drop(0).reset_index(drop=True)
df = df.rename(columns={df.columns[0]: 'Date'})

# keeping only real quarter rows
months = ['March', 'June', 'September', 'December']
df = df[df['Date'].apply(lambda x: any(m in str(x) for m in months))].copy()
df = df.iloc[:20].copy()

print("shape:", df.shape)
print("dates:", df['Date'].tolist())

shape: (20, 21)
dates: ['March 2021', 'June 2021', 'September 2021', 'December 2021', 'March 2022', 'June 2022', 'September 2022', 'December 2022', 'March 2023', 'June 2023', 'September 2023', 'December 2023', 'March 2024', 'June 2024', 'September 2024', 'December 2024', 'March 2025', 'June 2025', 'September 2025', 'December 2025']


d:\Users\Admin\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [2]:
# keeping only my 5 dissertation sectors
my_sectors = {
    'Technology': 'J : Information and communication',
    'Healthcare': 'Q : Human health and social work activities',
    'Finance': 'K : Financial and insurance activities',
    'Engineering': 'F : Construction',
    'Education': 'P : Education'
}

# keeping only date + my 5 sector columns
cols_to_keep = ['Date'] + list(my_sectors.values())
df_clean = df[cols_to_keep].copy()

# renaming columns to simple sector names
df_clean.columns = ['Date'] + list(my_sectors.keys())

# converting to numeric
for col in my_sectors.keys():
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

print("shape:", df_clean.shape)
print("\nfirst 5 rows:")
print(df_clean.head())
print("\nmissing values:")
print(df_clean.isnull().sum())

shape: (20, 6)

first 5 rows:
             Date  Technology  Healthcare  Finance  Engineering  Education
0      March 2021     1514194     4564464  1155306      2171301    2934152
1       June 2021     1497237     4582653  1163905      2238909    2941709
2  September 2021     1534787     4601135  1160384      2226359    2943473
3   December 2021     1564133     4591794  1141248      2223512    2978573
4      March 2022     1573034     4614980  1114519      2255084    3023351

missing values:
Date           0
Technology     0
Healthcare     0
Finance        0
Engineering    0
Education      0
dtype: int64


In [3]:
# standardising date to YYYY-Q format
def convert_date(date_str):
    month_to_q = {
        'March': 'Q1', 'June': 'Q2',
        'September': 'Q3', 'December': 'Q4'
    }
    for month, q in month_to_q.items():
        if month in str(date_str):
            year = str(date_str).split(' ')[-1]
            return f"{year}-{q}"
    return None

df_clean['Quarter'] = df_clean['Date'].apply(convert_date)
df_clean = df_clean.drop(columns=['Date'])
cols = ['Quarter'] + list(my_sectors.keys())
df_clean = df_clean[cols]

print("cleaned date format:")
print(df_clean['Quarter'].tolist())

cleaned date format:
['2021-Q1', '2021-Q2', '2021-Q3', '2021-Q4', '2022-Q1', '2022-Q2', '2022-Q3', '2022-Q4', '2023-Q1', '2023-Q2', '2023-Q3', '2023-Q4', '2024-Q1', '2024-Q2', '2024-Q3', '2024-Q4', '2025-Q1', '2025-Q2', '2025-Q3', '2025-Q4']


In [4]:
# saving the clean file
df_clean.to_csv('NOMIS_Vacancies_Clean.csv', index=False)
print("saved! shape:", df_clean.shape)
print("\nfinal preview:")
print(df_clean.head())

saved! shape: (20, 6)

final preview:
   Quarter  Technology  Healthcare  Finance  Engineering  Education
0  2021-Q1     1514194     4564464  1155306      2171301    2934152
1  2021-Q2     1497237     4582653  1163905      2238909    2941709
2  2021-Q3     1534787     4601135  1160384      2226359    2943473
3  2021-Q4     1564133     4591794  1141248      2223512    2978573
4  2022-Q1     1573034     4614980  1114519      2255084    3023351
